In [27]:
import pandas as pd 
import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from konlpy.tag import Komoran
from collections import Counter
# tqdm : 진행 상태를 로그로 표시하는 기능
from tqdm import tqdm

In [96]:
df = pd.read_csv("../data/ratings_train.txt", sep='\t')

df.dropna(inplace=True)

df.drop_duplicates('document', inplace=True)

df = df[:10000]

len(df)

10000

In [97]:
df['label'].value_counts()

label
0    5027
1    4973
Name: count, dtype: int64

In [98]:
komoran = Komoran()
# 모든 품사를 이용하니 학습의 능력이 떨어진다. 
# tokenized_sentence = [komoran.morphs(text) for text in df['document']]
# 품사를 필터링 
def tokenize(text):
    allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL']
    result = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            result.append(word)
    return result

tokenized_sentence = [ tokenize(text) for text in df['document'] ]

In [99]:
# 단어 사전을 생성 
# 패딩 토큰, 언노운 토큰 생성 (초기 값)
vocab = {
    "<PAD>" : 0, 
    "<UNK>" : 1
}
# tokenized_sentence에서 모든 토큰을 하나의 리스트로 생성 
all_tokens = [ token for tokens in tokenized_sentence for token in tokens ]
# token들의 빈도수를 확인 -> min_count로 제한 
token_counts = Counter(all_tokens)
token_counts

Counter({'영화': 3560,
         '보': 2716,
         '없': 1076,
         '하': 1067,
         '좋': 704,
         '있': 700,
         '정말': 670,
         '너무': 642,
         '안': 574,
         '같': 574,
         '재밌': 561,
         '진짜': 533,
         '만들': 513,
         '연기': 494,
         '나오': 475,
         '평점': 415,
         '최고': 414,
         '잘': 409,
         '되': 409,
         '왜': 398,
         '때': 372,
         '다': 365,
         '스토리': 353,
         '사람': 349,
         '드라마': 339,
         '이': 330,
         '알': 325,
         '말': 319,
         '배우': 317,
         '생각': 313,
         '감동': 311,
         '감독': 301,
         '더': 297,
         '아깝': 289,
         '내용': 289,
         '그냥': 282,
         '시간': 280,
         '나': 277,
         '!!': 275,
         '이렇': 266,
         '재미없': 258,
         '좀': 258,
         '재미': 246,
         '재미있': 243,
         '가': 236,
         '작품': 224,
         '모르': 219,
         '쓰레기': 217,
         '재': 214,
         '남': 204,
         '다시

In [100]:
# 단어의 빈도수가 3이상인 토큰들만을 이용하여 단어 사전에 넣어준다. 
for token, count in token_counts.items():
    if count >= 3:
        vocab[token] = len(vocab)

In [101]:
vocab

{'<PAD>': 0,
 '<UNK>': 1,
 '더빙': 2,
 '진짜': 3,
 '짜증': 4,
 '나': 5,
 '목소리': 6,
 '포스터': 7,
 '초딩': 8,
 '영화': 9,
 '오버': 10,
 '연기': 11,
 '가볍': 12,
 '이야기': 13,
 '솔직히': 14,
 '재미': 15,
 '없': 16,
 '평점': 17,
 '조정': 18,
 '돋보이': 19,
 '스파이더맨': 20,
 '늙': 21,
 '보이': 22,
 '하': 23,
 '너무나': 24,
 '막': 25,
 '떼': 26,
 '초등학교': 27,
 '학년': 28,
 '아깝': 29,
 '원작': 30,
 '긴장감': 31,
 '제대로': 32,
 '살리': 33,
 '반개': 34,
 '욕': 35,
 '나오': 36,
 '생활': 37,
 '이': 38,
 '정말': 39,
 '발로': 40,
 '납치': 41,
 '반복': 42,
 '드라마': 43,
 '가족': 44,
 '못하': 45,
 '사람': 46,
 '모이': 47,
 '액션': 48,
 '있': 49,
 '안': 50,
 '왜': 51,
 '낮': 52,
 '꽤': 53,
 '보': 54,
 '헐리우드': 55,
 '너무': 56,
 '볼': 57,
 '때': 58,
 '눈물': 59,
 '나서': 60,
 '죽': 61,
 '향수': 62,
 '자극': 63,
 '!!': 64,
 '감성': 65,
 '절제': 66,
 '멜로': 67,
 '달인': 68,
 '이다': 69,
 '울': 70,
 '드럽': 71,
 '좋': 72,
 '기사': 73,
 '로만': 74,
 '보다': 75,
 '자꾸': 76,
 '잊어버리': 77,
 '취향': 78,
 '존중': 79,
 '극장': 80,
 '가장': 81,
 '노': 82,
 '재': 83,
 '감동': 84,
 '스토리': 85,
 '어거지': 86,
 '매번': 87,
 '긴장': 88,
 '참': 89,
 '웃기': 90,
 '바스코

In [102]:
# vocab을 이용한 토큰화 된 데이터의 인코딩과 Dataset을 결합 
# dict.get() -> 특정 키를 입력하면 해당 키의 값을 되돌려주는 함수
# ( 두번째 인자값을 이용하여 첫번째 인자의 키 값이 존재하지 않을때 디폴트 값을 설정 )
vocab.get('마케팅', vocab['<UNK>'])

1

In [103]:
# Dataset을 선언
class RNNDataset(Dataset):
    # 생성자, 길이 출력함수, 특정위치의 데이터 출력함수 
    def __init__( self, tokenized_texts, labels, vocab ):
        # tokenized_texts : 토큰화된 문서들 (독립 변수)
        # labels : 정답 데이터 (종속 변수)
        # vocab : 단어 사전 
        self.labels = labels.values
        self.data = [
            [
                vocab.get(token, vocab['<UNK>']) for token in tokens
            ]
            for tokens in tokenized_texts
        ]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        # getitem 함수의 역할 : DataLoader가 데이터를 불러오는 함수 (독립 변수, 종속 변수)
        return torch.tensor(self.data[idx], dtype=torch.long), \
            torch.tensor(self.labels[idx], dtype=torch.long)

In [104]:
# 후 처리 가공 함수 (DataLoader가 배치 사이즈만큼 Dataset을 불러온 후 처리 가공)
def collate_fn(batch):
    # 배치 단위로 들어온 데이터를 최대 길이의 data에 맞게 패딩 토큰을 채워준다. 
    # 배치 -> [ (data, label), (data, label), ... ]
    text_list = [item[0] for item in batch]
    label_list = [item[1] for item in batch]

    # text_list에 있는 인코딩된 데이터에서 최대 길이만큼 나머지 데이터에 패딩 토큰을 채워준다. 
    padded_texts = pad_sequence(text_list, batch_first=True, padding_value=vocab['<PAD>'])
    labels = torch.tensor(label_list, dtype = torch.long)

    return padded_texts, labels

In [105]:
# Dataset 생성 
dataset = RNNDataset(tokenized_sentence, df['label'], vocab)
# train의 길이와 test의 길이를 설정 
train_size = int(len(dataset) * 0.8)   # int() 사용하는 이유는? 길이를 의미하기 때문에 정수형으로 변환(버림)
test_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, test_size])

In [106]:
print(len(train_dataset), len(val_dataset))

8000 2000


In [107]:
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True, collate_fn= collate_fn)
val_loader = DataLoader(val_dataset, batch_size = 64, shuffle=True, collate_fn=collate_fn)

In [108]:
class RNNCLF(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_classes):
        # vocab_size : 임베딩 함수 입력 차원의 수 
        # emb_dim : 임베딜 함수 출력 차원의 수 
        # hidden_size : RNN 은닉층의 출력 차원의 수
        # num_classes : 선형 모델의 출력 차원의 수 (분류 개수) 
        super().__init__()
        # 입력되는 데이터는 인코딩 된 데이터 (2,3,4) -> 벡터화 작업 ( nn.Enbedding(), Word2Vec, FastText, Doc2Vec )
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=vocab['<PAD>'])
        # RNN 모델 
        self.rnn = nn.RNN(emb_dim, hidden_size, batch_first=True)
        # 선형 모델 
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # x : DataLoader의 독립 변수 값(토큰화 데이터)
        embedding = self.emb(x)  # [batch_size, seq_len, emb_dim]

        # rnn_out -> [batch_size, seq_len, hidden_size] (모든 시점의 출력)
        # hidden -> [1, seq_len, hidden_size] (제일 마지막 시점의 은닉 상태)
        rnn_out, hidden = self.rnn(embedding)

        # 선형 모델에 데이터를 대입 하기 위해서 hidden의 배치층을 제거 
        last_hidden = hidden.squeeze(0)  # [seq_len, hidden]

        return self.fc(last_hidden)

In [109]:
# 모델 생성 
model = RNNCLF(len(vocab), emb_dim=64, hidden_size=128, num_classes=2)
# 손실 함수 
criterion = nn.CrossEntropyLoss()
# 옵티마이저 생성 
optimtizer = optim.Adam(model.parameters(), lr = 0.001)


In [110]:
epochs = 50

for epoch in range(epochs):
    model.train()
    train_loss = 0
    corret_train = 0
    total_train = 0
    # tqdm() -> desc는 로그 출력 값
    for inputs, labels in tqdm(train_loader, desc = f"Epoch {epoch+1} / {epochs} Train"):
        optimtizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, labels)
        loss.backward()
        optimtizer.step()

        train_loss += loss.item()
        pred = torch.argmax(output, dim=1)
        corret_train += (pred == labels).sum().item()
        total_train += labels.size(0)
    
    train_acc = (corret_train / total_train) * 100
    avg_train_loss = train_loss / len(train_loader)

    # 검증 구간 
    model.eval()
    val_loss = 0
    corret_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            output = model(inputs)
            loss = criterion(output, labels)

            val_loss += loss.item()
            pred = torch.argmax(output, dim=1)
            corret_val += (pred == labels).sum().item()
            total_val += labels.size(0)
    val_acc = (corret_val / total_val) * 100
    avg_val_loss = val_loss / len(val_loader)
    if (epoch + 1) % 10 == 0:
        print(f"RNN 에폭 : Train Loss : {round(avg_train_loss, 4)} Train Acc : {train_acc}")
        print(f"RNN 에폭 : Vali Loss : {round(avg_val_loss, 4)} Vali Acc : {val_acc}")



Epoch 10 / 50 Train: 100%|██████████| 125/125 [00:01<00:00, 121.87it/s]


RNN 에폭 : Train Loss : 0.69 Train Acc : 50.5875
RNN 에폭 : Vali Loss : 0.6984 Vali Acc : 51.0


Epoch 20 / 50 Train: 100%|██████████| 125/125 [00:01<00:00, 106.65it/s]


RNN 에폭 : Train Loss : 0.685 Train Acc : 54.974999999999994
RNN 에폭 : Vali Loss : 0.687 Vali Acc : 54.65


Epoch 30 / 50 Train: 100%|██████████| 125/125 [00:01<00:00, 101.46it/s]


RNN 에폭 : Train Loss : 0.6828 Train Acc : 55.400000000000006
RNN 에폭 : Vali Loss : 0.6903 Vali Acc : 54.800000000000004


Epoch 40 / 50 Train: 100%|██████████| 125/125 [00:01<00:00, 97.43it/s]


RNN 에폭 : Train Loss : 0.6846 Train Acc : 54.425000000000004
RNN 에폭 : Vali Loss : 0.6927 Vali Acc : 54.85


Epoch 50 / 50 Train: 100%|██████████| 125/125 [00:01<00:00, 98.16it/s] 


RNN 에폭 : Train Loss : 0.6829 Train Acc : 55.2125
RNN 에폭 : Vali Loss : 0.6892 Vali Acc : 56.10000000000001
